# Sampling Design States: Validation with TABVASTESTDATABASE

This notebook validates ProcessBehavior's handling of all six Sampling Design States (SDS 1-6) using the TABVASTESTDATABASE -- a 4000-row dataset designed to exercise every SDS through different response columns.

## The TABVASTESTDATABASE

The dataset has:
- **FACTOR 1**: 4 levels (1, 2, 3, 4)
- **FACTOR 2**: 2 levels (1, 2)
- **PRODUCTION TIME**: 100 time periods
- **PM SDS 1-6**: Response columns, each with a sparsity pattern matching its SDS

## `factors` vs `plan`

ProcessBehavior can detect all six SDS types using either `factors` or `plan`. SDS detection runs on raw data before NA rows are dropped, so cells where all response values are NA are counted as empty (N_kt=0) -- enabling SDS 4-6 detection even without a plan.

| Approach | What ProcessBehavior Knows | Capabilities |
|----------|---------------------------|----------------------------------|
| **`factors`** | Observed structure (including all-NA cells) | All SDS detection (1-6), charts, basic design reports |
| **`plan`** | Expected + observed structure | All SDS detection (1-6), coverage analysis, rich design reports |

## Overview

| SDS | Name | Structure | Recommended Chart |
|-----|------|-----------|-------------------|
| 1 | Full Replication | All cells n >= 2 | Xbar-S |
| 2 | No Replication | All cells n = 1 | IMR |
| 3 | Partial Replication | Mixed n=1 and n>=2 | Xbar-S (hybrid) |
| 4 | Incomplete with Singletons | Empty cells + mixed n | Xbar |
| 5 | Incomplete without Singletons | Empty cells + all n>=2 | Xbar |
| 6 | Incomplete without Replication | Empty cells + all n=1 | IMR |

In [ ]:
from processbehavior import ProcessBehavior

# Load the TABVASTESTDATABASE
pb = ProcessBehavior.read_csv('../../validation/TABVASTESTDATABASE.csv')

print(f"Dataset: {pb.data.shape[0]} rows x {pb.data.shape[1]} columns")
print(f"Columns: {pb.data.columns.tolist()}")

---
## SDS 1: Full Replication

Every (factor x time) cell has 2+ observations. This is the ideal structure for Wheeler's VAS.

**Capabilities:** Exact within-cell variance, all VAS residuals (R2-R5), full interaction analysis, Xbar-S with exact limits.

### Using `factors` (infer structure from data)

In [ ]:
# SDS 1 with factors -- let ProcessBehavior infer the structure
study_sds1 = pb.formulate(
    response='PM SDS 1',
    factors=['FACTOR 1', 'FACTOR 2'],
    time='PRODUCTION TIME',
    precision=3
)

print(f"SDS: {study_sds1.sds} - {study_sds1.sds_name}")
print(f"Valid charts: {study_sds1.valid_charts}")
print(f"Recommended: {study_sds1.recommended_chart}")

In [ ]:
# Design report shows observed structure (no plan specified)
print(study_sds1.design())

In [ ]:
# Execute Xbar+S paired -- returns both charts in one result
result_sds1 = study_sds1.execute(chart='Xbar', paired=True)
result_sds1.plot(chart='Xbar', show_stats=True, theme='ggplot').show()

In [ ]:
result_sds1.plot(chart='S', theme='ggplot').show()

### Using `plan` (specify expected structure)

The same data formulated with a plan provides richer design reporting -- showing planned vs observed structure.

In [ ]:
# SDS 1 with plan -- specify expected factor levels and time points
study_sds1_plan = pb.formulate(
    response='PM SDS 1',
    time='PRODUCTION TIME',
    plan={
        'factors': {
            'FACTOR 1': [1, 2, 3, 4],
            'FACTOR 2': [1, 2]
        },
        'T': 100,
        'N': 5
    },
    precision=3
)

print(f"SDS: {study_sds1_plan.sds} - {study_sds1_plan.sds_name}")
print()
# Design report now compares plan to observed
print(study_sds1_plan.design())

---
## SDS 2: No Replication

Every (factor x time) cell has exactly 1 observation. Common in automated measurement systems.

**Capabilities:** Variance estimated via 2-point moving average, IMR analysis.

In [ ]:
study_sds2 = pb.formulate(
    response=pb.cols.PM_SDS_2,
    factors=['FACTOR 1', 'FACTOR 2'],
    time='PRODUCTION TIME',
    precision=2
)

print(f"SDS: {study_sds2.sds} - {study_sds2.sds_name}")
print(f"Valid charts: {study_sds2.valid_charts}")
print()
print(study_sds2.design())

In [ ]:
# SDS 2 has n=1 per cell -- IMR works with individual observations
result_sds2 = study_sds2.execute(chart='Imr', by=['FACTOR 1', 'FACTOR 2'])
result_sds2.plot(chart='Imr', show_zones=True, theme='ggplot').show()

---
## SDS 3: Partial Replication

Mix of n=1 and n>=2 cells. The most common pattern in real-world data collection.

**Capabilities:** Hybrid variance estimation, Xbar-S with variable limits ("Varies" when N differs across subgroups).

In [ ]:
study_sds3 = pb.formulate(
    response=pb.cols.PM_SDS_3,
    factors=['FACTOR 1', 'FACTOR 2'],
    time='PRODUCTION TIME',
    precision=3
)

print(f"SDS: {study_sds3.sds} - {study_sds3.sds_name}")
print(f"Valid charts: {study_sds3.valid_charts}")
print()
print(study_sds3.design())

In [ ]:
# Xbar+S paired for SDS 3
result_sds3 = study_sds3.execute(chart='Xbar', paired=True)
result_sds3.plot(chart='Xbar', theme='ggplot').show()

In [ ]:
result_sds3.plot(chart='S', theme='ggplot').show()

---
## SDS 4-6: Incomplete Grids

SDS 4, 5, and 6 arise when the factor x time grid has empty cells -- combinations where all response values are NA or garbage. ProcessBehavior detects these automatically with either `factors` or `plan`.

Below we use `factors` to demonstrate that ProcessBehavior detects all three incomplete SDS types from the data alone. As shown in SDS 1 above, a `plan` can be added for richer design reporting.

### SDS 4: PM SDS 4 (using `factors`)

In [ ]:
study_sds4 = pb.formulate(
    response='PM SDS 4',
    factors=['FACTOR 1', 'FACTOR 2'],
    time='PRODUCTION TIME',
    precision=3
)

print(f"SDS: {study_sds4.sds} - {study_sds4.sds_name}")
print(f"Valid charts: {study_sds4.valid_charts}")
print()
print(study_sds4.design())

In [ ]:
result_sds4 = study_sds4.execute(chart='Xbar', paired=True)
result_sds4.plot(chart='Xbar', theme='ggplot').show()

### SDS 5: PM SDS 5 (using `factors`)

The PM SDS 5 column has even more missing data, creating a sparser grid.

In [ ]:
study_sds5 = pb.formulate(
    response='PM SDS 5',
    factors=['FACTOR 1', 'FACTOR 2'],
    time='PRODUCTION TIME',
    precision=3
)

print(f"SDS: {study_sds5.sds} - {study_sds5.sds_name}")
print(f"Valid charts: {study_sds5.valid_charts}")
print()
print(study_sds5.design())

In [ ]:
result_sds5 = study_sds5.execute(chart='Xbar', paired=True)
result_sds5.plot(chart='Xbar', theme='ggplot').show()

### SDS 6: PM SDS 6 (using `factors`)

The PM SDS 6 column has the most missing data -- the sparsest grid in the validation set.

In [ ]:
study_sds6 = pb.formulate(
    response='PM SDS 6',
    factors=['FACTOR 1', 'FACTOR 2'],
    time='PRODUCTION TIME',
    precision=3
)

print(f"SDS: {study_sds6.sds} - {study_sds6.sds_name}")
print(f"Valid charts: {study_sds6.valid_charts}")
print()
print(study_sds6.design())

In [ ]:
# SDS 6 recommends IMR due to sparse grid
result_sds6 = study_sds6.execute(chart='Imr', by=['FACTOR 1', 'FACTOR 2'])
result_sds6.plot(theme='ggplot').show()

---
## Chart Support Matrix

The `study.support` property shows which charts are available for each SDS, and why.

In [ ]:
# Compare chart availability across SDS types
for name, study in [('SDS 1', study_sds1), ('SDS 2', study_sds2), ('SDS 3', study_sds3)]:
    available = study.support[study.support['available']]['chart'].tolist()
    print(f"{name}: {available}")

---
## Working with Stratified Results

For stratified analyses, the `focus()` method lets you drill down to a single stratum.

In [ ]:
# Stratified IMR for SDS 1
result_stratified = study_sds1.execute(chart='Imr', by=['FACTOR 1'])

if result_stratified.is_stratified:
    print(f"Strata: {result_stratified.strata}")
    result_stratified.plot(chart='Imr', show_zones=True, theme='ggplot').show()

---
## Summary

### SDS Detection Summary

| SDS | Detection Basis | Plan Adds |
|-----|-----------------|----------|
| 1 | All cells n >= 2 | Design comparison, coverage reports |
| 2 | All cells n = 1 | Design comparison, coverage reports |
| 3 | Mixed cell sizes | Design comparison, shows sparse cells |
| 4 | Empty cells + mixed n | Catches entirely absent factor levels |
| 5 | Empty cells + all n>=2 | Catches entirely absent factor levels |
| 6 | Empty cells + all n=1 | Catches entirely absent factor levels |

### Key Takeaways

1. **Each PM column** in TABVASTESTDATABASE is designed to produce a specific SDS through its sparsity pattern
2. **SDS 4-6 detection works with `factors`** -- empty cells (all-NA responses) are detected from raw data
3. **`plan`** adds value by catching factor levels or time points entirely absent from the data
4. **`study.design()`** reveals planned vs observed gaps
5. **`study.support`** shows which charts are available and why
6. **Stratified results** support `focus()` for drilling into specific subgroups